# Chapter 14 — Recursion (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_java.ipynb)

This notebook goes with chapter 14 «Recursion» of the lecture notes (volume 4, *Analysis and recursion*) and with its [page of the site](https://fabiofurini.github.io/algorithms/recursion/14-recursion/). It contains the **Java** code of the algorithms with pseudo-code of the chapter: `Factorial`, `Fib`, `IterativeFib`, `RecursiveGCD`, `FastPower`, `Hanoi`. For every recursive algorithm the code counts the **calls** (the nodes of the tree of recursive calls), the **depth** of the recursion (the maximum height of the stack) and the **steps** (executions of the lines, numbered as in the lecture notes: line 1 is the `function` header), and prints the **trace** of the execution: the calls in depth-first order, with the indentation showing the position in the stack. The examples are those of the chapter.

**How to run it.** Click the «Open in Colab» badge, then *Runtime ▸ Run all* (or cell by cell with Shift+Enter). Nothing to install.

Every program is a `%%writefile` cell that writes the `.java` source file, followed by a cell that compiles it with `javac` and runs it with `java`. The first cell checks (or installs) the JDK.

**Names.** There are no arrays in this chapter: variables and parameters have the same names as in the pseudo-code. The stack and the counters are `static` fields of the class, and the trace is switched on with `trace = true`.

## Setup: the JDK

Colab already has `java`, but the compiler `javac` is needed too: the cell checks it and, if it is missing, installs the JDK.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## 1. Factorial

**Input:** a natural number $n$. **Output:** $n!$, with $0! = 1$. $S(n) = 2n + 2 = \Theta(n)$ steps; $n + 1$ calls, depth $n + 1$ (the tree is a path).

Pseudo-code: section 14.4 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [2]:
%%writefile Factorial.java
public class Factorial
{
static long[] cnt = new long[20];   // cnt[r] = number of executions of line r (numbering of the lecture notes)
static long calls = 0;              // number of calls = nodes of the tree of recursive calls
static int maxDepth = 0;            // depth of the recursion = maximum height of the stack
static long mults = 0;              // multiplications (FastPower)
static long moves = 0;              // moves (Hanoi)
static int ev = 0;                  // counter of the events of the trace
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // the call stack: the arguments of the active calls
static long[] seen = new long[64];  // seen[x] = number of calls with argument x
static boolean trace = false;       // true: print the trace of the execution

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Line | Executions");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Total steps S = " + total());
}

static void push(int x)             // a call starts: its frame goes on top of the stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // the call returns: its frame leaves the stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // numbered line of the trace, with the stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // line of the trace: the call, indented one level for every active call
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    // Factorial(n) correct only up to n = 20 (in C++ and Java 64 bits overflow beyond)
    static long factorial(int n)
    {
        push(n); cnt[2]++;
        if (n == 0)
        {
            cnt[3]++;
            if (trace) event("Factorial(0): n = 0 => base case");
            pop();
            if (trace) event("Factorial(0) returns 1");
            return 1;
        }
        if (trace) event(String.format("Factorial(%d): %d != 0 => must compute %d * Factorial(%d) and waits", n, n, n, n - 1));
        cnt[5]++;
        long f = factorial(n - 1);
        long r = n * f;
        pop();
        if (trace) event(String.format("Factorial(%d) returns %d * %d = %d", n, n, f, r));
        return r;
    }

    public static void main(String[] args)
    {
        // Example of the lecture notes: Factorial(4), with the stack
        int n = 4;
        reset();
        trace = true;
        long r = factorial(n);
        trace = false;
        System.out.printf("Factorial(%d) = %d\n", n, r);
        System.out.printf("Calls = %d (nodes of the tree), depth of the recursion = %d\n", calls, maxDepth);
        printSteps(2, 3, 5);
        System.out.printf("S(n) = 2n + 2 = %d\n", 2 * n + 2);

        // Calls, depth and steps as n varies
        System.out.printf("\n");
        System.out.printf("  n |                   n! |    calls |       depth | S(n) | 2n + 2\n");
        int[] values = {0, 1, 2, 3, 4, 5, 10, 15, 20};
        for (int v : values)
        {
            reset();
            long f = factorial(v);
            System.out.printf("%3d | %20d | %8d | %11d | %4d | %d\n", v, f, calls, maxDepth, total(), 2 * v + 2);
        }
    }
}

Writing Factorial.java


In [3]:
!javac Factorial.java && java Factorial

 1. Factorial(4): 4 != 0 => must compute 4 * Factorial(3) and waits   stack: [4]
 2. Factorial(3): 3 != 0 => must compute 3 * Factorial(2) and waits   stack: [4, 3]
 3. Factorial(2): 2 != 0 => must compute 2 * Factorial(1) and waits   stack: [4, 3, 2]
 4. Factorial(1): 1 != 0 => must compute 1 * Factorial(0) and waits   stack: [4, 3, 2, 1]
 5. Factorial(0): n = 0 => base case   stack: [4, 3, 2, 1, 0]
 6. Factorial(0) returns 1   stack: [4, 3, 2, 1]
 7. Factorial(1) returns 1 * 1 = 1   stack: [4, 3, 2]
 8. Factorial(2) returns 2 * 1 = 2   stack: [4, 3]
 9. Factorial(3) returns 3 * 2 = 6   stack: [4]
10. Factorial(4) returns 4 * 6 = 24   stack: []
Factorial(4) = 24
Calls = 5 (nodes of the tree), depth of the recursion = 5
Line | Executions
   2 |          5
   3 |          1
   5 |          4
Total steps S = 10
S(n) = 2n + 2 = 10

  n |                   n! |    calls |       depth | S(n) | 2n + 2
  0 |                    1 |        1 |           1 |    2 | 2
  1 |                    1 |

**Try it yourself.** Change $n$ and watch the stack grow and then empty.

In [4]:
%%writefile ProvaFactorial.java
public class ProvaFactorial
{
    public static void main(String[] args)
    {
        int n = 6;        // <-- change n (20 at most)
        Factorial.reset();
        Factorial.trace = true;
        long r = Factorial.factorial(n);
        Factorial.trace = false;
        System.out.printf("Factorial(%d) = %d, calls = %d, depth = %d, S = %d\n", n, r, Factorial.calls, Factorial.maxDepth, Factorial.total());
    }
}

Writing ProvaFactorial.java


In [5]:
!javac ProvaFactorial.java && java ProvaFactorial

 1. Factorial(6): 6 != 0 => must compute 6 * Factorial(5) and waits   stack: [6]
 2. Factorial(5): 5 != 0 => must compute 5 * Factorial(4) and waits   stack: [6, 5]
 3. Factorial(4): 4 != 0 => must compute 4 * Factorial(3) and waits   stack: [6, 5, 4]
 4. Factorial(3): 3 != 0 => must compute 3 * Factorial(2) and waits   stack: [6, 5, 4, 3]
 5. Factorial(2): 2 != 0 => must compute 2 * Factorial(1) and waits   stack: [6, 5, 4, 3, 2]
 6. Factorial(1): 1 != 0 => must compute 1 * Factorial(0) and waits   stack: [6, 5, 4, 3, 2, 1]
 7. Factorial(0): n = 0 => base case   stack: [6, 5, 4, 3, 2, 1, 0]
 8. Factorial(0) returns 1   stack: [6, 5, 4, 3, 2, 1]
 9. Factorial(1) returns 1 * 1 = 1   stack: [6, 5, 4, 3, 2]
10. Factorial(2) returns 2 * 1 = 2   stack: [6, 5, 4, 3]
11. Factorial(3) returns 3 * 2 = 6   stack: [6, 5, 4]
12. Factorial(4) returns 4 * 6 = 24   stack: [6, 5]
13. Factorial(5) returns 5 * 24 = 120   stack: [6]
14. Factorial(6) returns 6 * 120 = 720   stack: []
Factorial(6) = 720, c

## 2. Fibonacci numbers: Fib

**Input:** a natural number $n$. **Output:** $F(n)$, with $F(0) = 0$, $F(1) = 1$ and $F(n) = F(n-1) + F(n-2)$. Naive recursive algorithm: $C(n) = 2F(n+1) - 1$ calls and $S(n) = 4F(n+1) - 2 = 2^{\Theta(n)}$ steps.

Pseudo-code: section 14.5 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [6]:
%%writefile Fib.java
public class Fib
{
static long[] cnt = new long[20];   // cnt[r] = number of executions of line r (numbering of the lecture notes)
static long calls = 0;              // number of calls = nodes of the tree of recursive calls
static int maxDepth = 0;            // depth of the recursion = maximum height of the stack
static long mults = 0;              // multiplications (FastPower)
static long moves = 0;              // moves (Hanoi)
static int ev = 0;                  // counter of the events of the trace
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // the call stack: the arguments of the active calls
static long[] seen = new long[64];  // seen[x] = number of calls with argument x
static boolean trace = false;       // true: print the trace of the execution

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Line | Executions");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Total steps S = " + total());
}

static void push(int x)             // a call starts: its frame goes on top of the stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // the call returns: its frame leaves the stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // numbered line of the trace, with the stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // line of the trace: the call, indented one level for every active call
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    // Fib(n)
    static long fib(int n)
    {
        push(n); cnt[2]++;
        if (n <= 1)
        {
            cnt[3]++;
            if (trace) say(String.format("Fib(%d)", n), String.format("base case, returns %d", n));
            pop();
            return n;
        }
        if (trace) say(String.format("Fib(%d)", n), String.format("calls Fib(%d)", n - 1));
        long x = fib(n - 1);                       // the two calls are separated only to print the trace
        if (trace) say(String.format("Fib(%d)", n), String.format("calls Fib(%d)", n - 2));
        long y = fib(n - 2);
        cnt[5]++;
        if (trace) say(String.format("Fib(%d)", n), String.format("returns %d + %d = %d", x, y, x + y));
        pop();
        return x + y;
    }

    static long fibValue(int k)                    // F(k) with a loop, to check
    {
        long a = 0, b = 1;
        for (int i = 0; i < k; i++)
        {
            long t = a + b;
            a = b;
            b = t;
        }
        return a;
    }

    public static void main(String[] args)
    {
        // Example of the lecture notes: the execution of Fib(4), in depth-first order
        int n = 4;
        reset();
        trace = true;
        System.out.println(String.format("%-28s| %s", "call", "action"));
        long r = fib(n);
        trace = false;
        System.out.printf("F(%d) = %d, calls C(%d) = %d = 2F(%d) - 1, depth = %d\n", n, r, n, calls, n + 1, maxDepth);

        // Fib(5): the same subproblems are solved many times
        System.out.printf("\n");
        n = 5;
        reset();
        r = fib(n);
        System.out.printf("Fib(%d) = %d: %d calls, depth %d, S = %d steps\n", n, r, calls, maxDepth, total());
        System.out.printf("  k | calls of Fib(k)\n");
        for (int k = n; k >= 0; k--) System.out.printf("%3d | %d\n", k, seen[k]);
        printSteps(2, 3, 5);

        // Number of calls and steps: C(n) = 2F(n+1) - 1 and S(n) = 4F(n+1) - 2
        System.out.printf("\n");
        System.out.printf("  n |   F(n) |     C(n) | 2F(n+1) - 1 |     S(n) | 4F(n+1) - 2 | depth\n");
        int[] values = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30};
        for (int v : values)
        {
            reset();
            r = fib(v);
            System.out.printf("%3d | %6d | %8d | %11d | %8d | %11d | %d\n", v, r, calls, 2 * fibValue(v + 1) - 1, total(), 4 * fibValue(v + 1) - 2, maxDepth);
        }
    }
}

Writing Fib.java


In [7]:
!javac Fib.java && java Fib

call                        | action
Fib(4)                      | calls Fib(3)
  Fib(3)                    | calls Fib(2)
    Fib(2)                  | calls Fib(1)
      Fib(1)                | base case, returns 1
    Fib(2)                  | calls Fib(0)
      Fib(0)                | base case, returns 0
    Fib(2)                  | returns 1 + 0 = 1
  Fib(3)                    | calls Fib(1)
    Fib(1)                  | base case, returns 1
  Fib(3)                    | returns 1 + 1 = 2
Fib(4)                      | calls Fib(2)
  Fib(2)                    | calls Fib(1)
    Fib(1)                  | base case, returns 1
  Fib(2)                    | calls Fib(0)
    Fib(0)                  | base case, returns 0
  Fib(2)                    | returns 1 + 0 = 1
Fib(4)                      | returns 2 + 1 = 3
F(4) = 3, calls C(4) = 9 = 2F(5) - 1, depth = 4

Fib(5) = 5: 15 calls, depth 5, S = 30 steps
  k | calls of Fib(k)
  5 | 1
  4 | 1
  3 | 2
  2 | 3
  1 | 5
  0 | 3
Line | Ex

**Try it yourself.** Change $n$ and see how fast the calls grow.

In [8]:
%%writefile ProvaFib.java
public class ProvaFib
{
    public static void main(String[] args)
    {
        int n = 6;        // <-- change n (with n > 45 it takes a while)
        Fib.reset();
        long r = Fib.fib(n);
        System.out.printf("F(%d) = %d, calls = %d, S = %d\n", n, r, Fib.calls, Fib.total());
    }
}

Writing ProvaFib.java


In [9]:
!javac ProvaFib.java && java ProvaFib

F(6) = 8, calls = 25, S = 50


## 3. Iterative Fibonacci: IterativeFib

**Input:** a natural number $n$. **Output:** $F(n)$, computed from $F(0)$ and $F(1)$ upwards keeping only the last two values. $S(n) = 5n + 1 = \Theta(n)$ steps. Invariant: before every evaluation of the condition, $prev = F(i-2)$ and $curr = F(i-1)$.

Pseudo-code: section 14.5.2 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [10]:
%%writefile IterativeFib.java
public class IterativeFib
{
static long[] cnt = new long[20];   // cnt[r] = number of executions of line r (numbering of the lecture notes)
static long calls = 0;              // number of calls = nodes of the tree of recursive calls
static int maxDepth = 0;            // depth of the recursion = maximum height of the stack
static long mults = 0;              // multiplications (FastPower)
static long moves = 0;              // moves (Hanoi)
static int ev = 0;                  // counter of the events of the trace
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // the call stack: the arguments of the active calls
static long[] seen = new long[64];  // seen[x] = number of calls with argument x
static boolean trace = false;       // true: print the trace of the execution

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Line | Executions");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Total steps S = " + total());
}

static void push(int x)             // a call starts: its frame goes on top of the stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // the call returns: its frame leaves the stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // numbered line of the trace, with the stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // line of the trace: the call, indented one level for every active call
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    // IterativeFib(n)
    static long iterativeFib(int n, boolean trace)
    {
        cnt[2]++;
        if (n <= 1)
        {
            cnt[3]++;
            return n;
        }
        long prev = 0; cnt[5]++;
        long curr = 1; cnt[6]++;
        int i = 2; cnt[7]++;
        if (trace)                                 // trace: initialization row
            System.out.println(String.format("%-10s | %-8s | %-18s | %4d | %4d", "start", "2", "-", prev, curr));
        cnt[8]++;
        while (i <= n)
        {
            long next = prev + curr; cnt[9]++;
            prev = curr; cnt[10]++;
            curr = next; cnt[11]++;
            i = i + 1; cnt[12]++;
            cnt[8]++;
            if (trace)                             // trace: i - 2 is the number of the iteration
                System.out.println(String.format("%-10d | %-8s | %-18s | %4d | %4d", i - 2, String.format("%d -> %d", i - 1, i), String.format("%d + %d = %d", next - prev, prev, next), prev, curr));
        }
        cnt[14]++;
        return curr;
    }

    public static void main(String[] args)
    {
        // Example of the lecture notes: IterativeFib(4)
        int n = 4;
        reset();
        System.out.printf("iteration | i        | next = prev + curr | prev | curr\n");
        long r = iterativeFib(n, true);
        System.out.printf("The loop ends with i = %d > %d: it returns curr = %d\n", n + 1, n, r);
        printSteps(2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14);
        System.out.printf("S(n) = 5n + 1 = %d\n", 5 * n + 1);

        // Comparison with Fib: steps of Fib = 4F(n+1) - 2 (formula checked in the previous section)
        System.out.printf("\n");
        System.out.printf("  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1\n");
        int[] values = {5, 10, 20, 30};
        for (int v : values)
        {
            reset();
            r = iterativeFib(v, false);
            long steps = total();
            System.out.printf("%3d | %6d | %16d | %12d | %d\n", v, r, 4 * iterativeFib(v + 1, false) - 2, steps, 5 * v + 1);
        }
    }
}

Writing IterativeFib.java


In [11]:
!javac IterativeFib.java && java IterativeFib

iteration | i        | next = prev + curr | prev | curr
start      | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
The loop ends with i = 5 > 4: it returns curr = 3
Line | Executions
   2 |          1
   3 |          0
   5 |          1
   6 |          1
   7 |          1
   8 |          4
   9 |          3
  10 |          3
  11 |          3
  12 |          3
  14 |          1
Total steps S = 21
S(n) = 5n + 1 = 21

  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1
  5 |      5 |               30 |           26 | 26
 10 |     55 |              354 |           51 | 51
 20 |   6765 |            43782 |          101 | 101
 30 | 832040 |          5385074 |          151 | 151


**Try it yourself.** Change $n$ and check that the steps are $5n + 1$.

In [12]:
%%writefile ProvaIterativeFib.java
public class ProvaIterativeFib
{
    public static void main(String[] args)
    {
        int n = 10;        // <-- change n
        IterativeFib.reset();
        System.out.printf("iteration | i        | next = prev + curr | prev | curr\n");
        long r = IterativeFib.iterativeFib(n, true);
        System.out.printf("F(%d) = %d, S = %d = 5n + 1 = %d\n", n, r, IterativeFib.total(), 5 * n + 1);
    }
}

Writing ProvaIterativeFib.java


In [13]:
!javac ProvaIterativeFib.java && java ProvaIterativeFib

iteration | i        | next = prev + curr | prev | curr
start      | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
4          | 5 -> 6   | 2 + 3 = 5          |    3 |    5
5          | 6 -> 7   | 3 + 5 = 8          |    5 |    8
6          | 7 -> 8   | 5 + 8 = 13         |    8 |   13
7          | 8 -> 9   | 8 + 13 = 21        |   13 |   21
8          | 9 -> 10  | 13 + 21 = 34       |   21 |   34
9          | 10 -> 11 | 21 + 34 = 55       |   34 |   55
F(10) = 55, S = 51 = 5n + 1 = 51


## 4. Greatest common divisor: RecursiveGCD

**Input:** two natural numbers $n \ge 1$ and $m \ge 0$. **Output:** $\gcd(n, m)$. Tail recursion: each call makes at most one, the tree is a path. At most $2\lfloor\log_2 m\rfloor + 3$ calls, $O(\log_2 m)$ steps.

Pseudo-code: section 14.6 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [14]:
%%writefile RecursiveGCD.java
public class RecursiveGCD
{
static long[] cnt = new long[20];   // cnt[r] = number of executions of line r (numbering of the lecture notes)
static long calls = 0;              // number of calls = nodes of the tree of recursive calls
static int maxDepth = 0;            // depth of the recursion = maximum height of the stack
static long mults = 0;              // multiplications (FastPower)
static long moves = 0;              // moves (Hanoi)
static int ev = 0;                  // counter of the events of the trace
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // the call stack: the arguments of the active calls
static long[] seen = new long[64];  // seen[x] = number of calls with argument x
static boolean trace = false;       // true: print the trace of the execution

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Line | Executions");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Total steps S = " + total());
}

static void push(int x)             // a call starts: its frame goes on top of the stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // the call returns: its frame leaves the stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // numbered line of the trace, with the stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // line of the trace: the call, indented one level for every active call
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    static int floorLog2(int m)                    // floor(log2 m) with a loop (only for the bound on the calls)
    {
        int k = 0;
        while (m > 1)
        {
            m = m / 2;
            k = k + 1;
        }
        return k;
    }

    // RecursiveGCD(n, m)
    static int recursiveGCD(int n, int m)
    {
        push(m); cnt[2]++;
        if (m == 0)
        {
            cnt[3]++;
            if (trace) say(String.format("RecursiveGCD(%d, %d)", n, m), String.format("base case, returns %d", n));
            pop();
            return n;
        }
        if (trace) say(String.format("RecursiveGCD(%d, %d)", n, m), String.format("%d mod %d = %d: calls RecursiveGCD(%d, %d)", n, m, n % m, m, n % m));
        cnt[5]++;
        int g = recursiveGCD(m, n % m);
        if (trace) say(String.format("RecursiveGCD(%d, %d)", n, m), String.format("returns %d", g));
        pop();
        return g;
    }

    public static void main(String[] args)
    {
        // Examples of the lecture notes: RecursiveGCD(48, 18) and RecursiveGCD(252, 105), trees of calls (paths)
        int[][] pairs = {{48, 18}, {252, 105}};
        for (int[] p : pairs)
        {
            reset();
            trace = true;
            System.out.println(String.format("%-28s| %s", "call", "action"));
            int g = recursiveGCD(p[0], p[1]);
            trace = false;
            System.out.printf("gcd(%d, %d) = %d, calls = %d, S = %d\n", p[0], p[1], g, calls, total());
            System.out.printf("\n");
        }

        // Number of calls (each call executes 2 lines, hence S = 2 calls) and bound 2 floor(log2 m) + 3
        System.out.printf("      n |       m |  gcd |    calls |   S | 2 floor(log2 m) + 3\n");
        int[][] more = {{48, 18}, {252, 105}, {89, 55}, {144, 89}, {1000, 7}, {1000000, 999999}};
        for (int[] p : more)
        {
            reset();
            int g = recursiveGCD(p[0], p[1]);
            System.out.printf("%7d | %7d | %4d | %8d | %3d | %d\n", p[0], p[1], g, calls, total(), 2 * floorLog2(p[1]) + 3);
        }
    }
}

Writing RecursiveGCD.java


In [15]:
!javac RecursiveGCD.java && java RecursiveGCD

call                        | action
RecursiveGCD(48, 18)        | 48 mod 18 = 12: calls RecursiveGCD(18, 12)
  RecursiveGCD(18, 12)      | 18 mod 12 = 6: calls RecursiveGCD(12, 6)
    RecursiveGCD(12, 6)     | 12 mod 6 = 0: calls RecursiveGCD(6, 0)
      RecursiveGCD(6, 0)    | base case, returns 6
    RecursiveGCD(12, 6)     | returns 6
  RecursiveGCD(18, 12)      | returns 6
RecursiveGCD(48, 18)        | returns 6
gcd(48, 18) = 6, calls = 4, S = 8

call                        | action
RecursiveGCD(252, 105)      | 252 mod 105 = 42: calls RecursiveGCD(105, 42)
  RecursiveGCD(105, 42)     | 105 mod 42 = 21: calls RecursiveGCD(42, 21)
    RecursiveGCD(42, 21)    | 42 mod 21 = 0: calls RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | base case, returns 21
    RecursiveGCD(42, 21)    | returns 21
  RecursiveGCD(105, 42)     | returns 21
RecursiveGCD(252, 105)      | returns 21
gcd(252, 105) = 21, calls = 4, S = 8

      n |       m |  gcd |    calls |   S | 2 floor(log2 m) + 3
     48 |

**Try it yourself.** Change the two numbers; the worst case is two consecutive Fibonacci numbers.

In [16]:
%%writefile ProvaRecursiveGCD.java
public class ProvaRecursiveGCD
{
    public static void main(String[] args)
    {
        int n = 1071, m = 462;        // <-- change the two numbers
        RecursiveGCD.reset();
        RecursiveGCD.trace = true;
        System.out.println(String.format("%-28s| %s", "call", "action"));
        int g = RecursiveGCD.recursiveGCD(n, m);
        RecursiveGCD.trace = false;
        System.out.printf("gcd(%d, %d) = %d, calls = %d\n", n, m, g, RecursiveGCD.calls);
    }
}

Writing ProvaRecursiveGCD.java


In [17]:
!javac ProvaRecursiveGCD.java && java ProvaRecursiveGCD

call                        | action
RecursiveGCD(1071, 462)     | 1071 mod 462 = 147: calls RecursiveGCD(462, 147)
  RecursiveGCD(462, 147)    | 462 mod 147 = 21: calls RecursiveGCD(147, 21)
    RecursiveGCD(147, 21)   | 147 mod 21 = 0: calls RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | base case, returns 21
    RecursiveGCD(147, 21)   | returns 21
  RecursiveGCD(462, 147)    | returns 21
RecursiveGCD(1071, 462)     | returns 21
gcd(1071, 462) = 21, calls = 4


## 5. Fast exponentiation: FastPower

**Input:** a number $x$ and a natural number $n$. **Output:** $x^n$, by repeated squaring. $S(n) = 4\lfloor\log_2 n\rfloor + 6 = \Theta(\log_2 n)$ steps, $\lfloor\log_2 n\rfloor + 2$ calls, at most $2(\lfloor\log_2 n\rfloor + 1)$ multiplications.

Pseudo-code: section 14.7 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [18]:
%%writefile FastPower.java
public class FastPower
{
static long[] cnt = new long[20];   // cnt[r] = number of executions of line r (numbering of the lecture notes)
static long calls = 0;              // number of calls = nodes of the tree of recursive calls
static int maxDepth = 0;            // depth of the recursion = maximum height of the stack
static long mults = 0;              // multiplications (FastPower)
static long moves = 0;              // moves (Hanoi)
static int ev = 0;                  // counter of the events of the trace
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // the call stack: the arguments of the active calls
static long[] seen = new long[64];  // seen[x] = number of calls with argument x
static boolean trace = false;       // true: print the trace of the execution

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Line | Executions");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Total steps S = " + total());
}

static void push(int x)             // a call starts: its frame goes on top of the stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // the call returns: its frame leaves the stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // numbered line of the trace, with the stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // line of the trace: the call, indented one level for every active call
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    static int floorLog2(int n)                    // floor(log2 n) with a loop (only for the formula of the steps)
    {
        int k = 0;
        while (n > 1)
        {
            n = n / 2;
            k = k + 1;
        }
        return k;
    }

    // FastPower(x, n)   values that are too large overflow in C++ and Java, not in Python
    static long fastPower(long x, int n)
    {
        push(n); cnt[2]++;
        if (n == 0)
        {
            cnt[3]++;
            if (trace) say(String.format("FastPower(%d, 0)", x), "base case, returns 1");
            pop();
            return 1;
        }
        if (trace) say(String.format("FastPower(%d, %d)", x, n), String.format("%d != 0 => calls FastPower(%d, %d)   (%d/2 = %d)", n, x, n / 2, n, n / 2));
        long y = fastPower(x, n / 2); cnt[5]++;    // one recursive call only: the result is in y
        cnt[6]++;
        long r;
        if (n % 2 == 0)
        {
            cnt[7]++;
            mults += 1;
            r = y * y;
            if (trace) say(String.format("FastPower(%d, %d)", x, n), String.format("y = %d, %d is even => returns %d * %d = %d", y, n, y, y, r));
        }
        else
        {
            cnt[9]++;
            mults += 2;
            r = x * y * y;
            if (trace) say(String.format("FastPower(%d, %d)", x, n), String.format("y = %d, %d is odd => returns %d * %d * %d = %d", y, n, x, y, y, r));
        }
        pop();
        return r;
    }

    public static void main(String[] args)
    {
        // Example of the lecture notes: FastPower(3, 13)
        long x = 3;
        int n = 13;
        reset();
        trace = true;
        System.out.println(String.format("%-28s| %s", "call", "action"));
        long r = fastPower(x, n);
        trace = false;
        System.out.printf("%d^%d = %d, calls = %d, multiplications = %d\n", x, n, r, calls, mults);
        printSteps(2, 3, 5, 6, 7, 9);
        System.out.printf("S(n) = 4 floor(log2 n) + 6 = %d\n", 4 * floorLog2(n) + 6);

        // How many multiplications: about 2 log2 n instead of n (x = 1, to avoid overflow)
        System.out.printf("\n");
        System.out.printf("      n |    calls | multiplications | S(n) | 4 floor(log2 n) + 6\n");
        int[] values = {1, 2, 13, 100, 1000, 1000000};
        for (int v : values)
        {
            reset();
            fastPower(1, v);
            System.out.printf("%7d | %8d | %15d | %4d | %d\n", v, calls, mults, total(), 4 * floorLog2(v) + 6);
        }
    }
}

Writing FastPower.java


In [19]:
!javac FastPower.java && java FastPower

call                        | action
FastPower(3, 13)            | 13 != 0 => calls FastPower(3, 6)   (13/2 = 6)
  FastPower(3, 6)           | 6 != 0 => calls FastPower(3, 3)   (6/2 = 3)
    FastPower(3, 3)         | 3 != 0 => calls FastPower(3, 1)   (3/2 = 1)
      FastPower(3, 1)       | 1 != 0 => calls FastPower(3, 0)   (1/2 = 0)
        FastPower(3, 0)     | base case, returns 1
      FastPower(3, 1)       | y = 1, 1 is odd => returns 3 * 1 * 1 = 3
    FastPower(3, 3)         | y = 3, 3 is odd => returns 3 * 3 * 3 = 27
  FastPower(3, 6)           | y = 27, 6 is even => returns 27 * 27 = 729
FastPower(3, 13)            | y = 729, 13 is odd => returns 3 * 729 * 729 = 1594323
3^13 = 1594323, calls = 5, multiplications = 7
Line | Executions
   2 |          5
   3 |          1
   5 |          4
   6 |          4
   7 |          1
   9 |          3
Total steps S = 18
S(n) = 4 floor(log2 n) + 6 = 18

      n |    calls | multiplications | S(n) | 4 floor(log2 n) + 6
      1 |        2 |   

**Try it yourself.** Change $x$ and $n$ and count the multiplications.

In [20]:
%%writefile ProvaFastPower.java
public class ProvaFastPower
{
    public static void main(String[] args)
    {
        long x = 2;
        int n = 10;        // <-- change x and n (with large x and n C++ and Java overflow)
        FastPower.reset();
        FastPower.trace = true;
        System.out.println(String.format("%-28s| %s", "call", "action"));
        long r = FastPower.fastPower(x, n);
        FastPower.trace = false;
        System.out.printf("%d^%d = %d, multiplications = %d\n", x, n, r, FastPower.mults);
    }
}

Writing ProvaFastPower.java


In [21]:
!javac ProvaFastPower.java && java ProvaFastPower

call                        | action
FastPower(2, 10)            | 10 != 0 => calls FastPower(2, 5)   (10/2 = 5)
  FastPower(2, 5)           | 5 != 0 => calls FastPower(2, 2)   (5/2 = 2)
    FastPower(2, 2)         | 2 != 0 => calls FastPower(2, 1)   (2/2 = 1)
      FastPower(2, 1)       | 1 != 0 => calls FastPower(2, 0)   (1/2 = 0)
        FastPower(2, 0)     | base case, returns 1
      FastPower(2, 1)       | y = 1, 1 is odd => returns 2 * 1 * 1 = 2
    FastPower(2, 2)         | y = 2, 2 is even => returns 2 * 2 = 4
  FastPower(2, 5)           | y = 4, 5 is odd => returns 2 * 4 * 4 = 32
FastPower(2, 10)            | y = 32, 10 is even => returns 32 * 32 = 1024
2^10 = 1024, multiplications = 6


## 6. Towers of Hanoi: Hanoi

**Input:** the number of disks $n \ge 1$, the source peg $from$, the destination peg $to$ and the auxiliary peg $aux$. **Output:** the sequence of moves that transfers disks $1, \dots, n$ from $from$ to $to$. $M(n) = 2^n - 1$ moves (optimal), $S(n) = 3 \cdot 2^n - 4 = \Theta(2^n)$ steps, depth $n$.

Pseudo-code: section 14.8 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [22]:
%%writefile Hanoi.java
public class Hanoi
{
static long[] cnt = new long[20];   // cnt[r] = number of executions of line r (numbering of the lecture notes)
static long calls = 0;              // number of calls = nodes of the tree of recursive calls
static int maxDepth = 0;            // depth of the recursion = maximum height of the stack
static long mults = 0;              // multiplications (FastPower)
static long moves = 0;              // moves (Hanoi)
static int ev = 0;                  // counter of the events of the trace
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // the call stack: the arguments of the active calls
static long[] seen = new long[64];  // seen[x] = number of calls with argument x
static boolean trace = false;       // true: print the trace of the execution

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Line | Executions");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Total steps S = " + total());
}

static void push(int x)             // a call starts: its frame goes on top of the stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // the call returns: its frame leaves the stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // numbered line of the trace, with the stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // line of the trace: the call, indented one level for every active call
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    static boolean showMoves = false;              // true: print the moves

    static void move(int disk, String from, String to)   // "move disk d from from to to": counts the move and, if showMoves, prints it
    {
        moves++;
        if (showMoves) System.out.println(String.format("%d. move disk %d from %s to %s", moves, disk, from, to));
    }

    // Hanoi(n, from, to, aux)
    static void hanoi(int n, String from, String to, String aux)
    {
        push(n); cnt[2]++;
        String call = trace ? String.format("Hanoi(%d,%s,%s,%s)", n, from, to, aux) : "";
        if (n == 1)
        {
            cnt[3]++;
            move(1, from, to);
            if (trace) say(call, String.format("prints: move disk 1 from %s to %s, returns", from, to));
        }
        else
        {
            if (trace) say(call, String.format("calls Hanoi(%d,%s,%s,%s)", n - 1, from, aux, to));
            cnt[5]++;
            hanoi(n - 1, from, aux, to);
            cnt[6]++;
            move(n, from, to);
            if (trace) say(call, String.format("prints: move disk %d from %s to %s, calls Hanoi(%d,%s,%s,%s)", n, from, to, n - 1, aux, to, from));
            cnt[7]++;
            hanoi(n - 1, aux, to, from);
            if (trace) say(call, "returns");
        }
        pop();
    }

    public static void main(String[] args)
    {
        // Example of the lecture notes: Hanoi(3, A, C, B), the moves
        reset();
        showMoves = true;
        hanoi(3, "A", "C", "B");
        showMoves = false;
        System.out.printf("%d moves, %d calls, depth of the recursion = %d\n", moves, calls, maxDepth);

        // The same execution, call by call (indentation = depth in the stack)
        System.out.printf("\n");
        reset();
        trace = true;
        System.out.println(String.format("%-28s| %s", "call", "action"));
        hanoi(3, "A", "C", "B");
        trace = false;
        printSteps(2, 3, 5, 6, 7);
        System.out.printf("S(n) = 3 * 2^n - 4 = %d\n", 3 * (1 << 3) - 4);

        // Moves and steps: M(n) = 2^n - 1 and S(n) = 3 * 2^n - 4
        System.out.printf("\n");
        System.out.printf("  n |  moves | 2^n - 1 |    calls |   S(n) | 3 * 2^n - 4 | depth\n");
        for (int k = 1; k <= 10; k++)
        {
            reset();
            hanoi(k, "A", "C", "B");
            System.out.printf("%3d | %6d | %7d | %8d | %6d | %11d | %d\n", k, moves, (1 << k) - 1, calls, total(), 3 * (1 << k) - 4, maxDepth);
        }

        // The legend: 64 disks, one move per second
        System.out.printf("\n");
        System.out.printf("n = 64: 2^64 - 1 seconds are more than %.1e years\n", (Math.pow(2.0, 64) - 1) / (365.25 * 24 * 3600));
    }
}

Writing Hanoi.java


In [23]:
!javac Hanoi.java && java Hanoi

1. move disk 1 from A to C
2. move disk 2 from A to B
3. move disk 1 from C to B
4. move disk 3 from A to C
5. move disk 1 from B to A
6. move disk 2 from B to C
7. move disk 1 from A to C
7 moves, 7 calls, depth of the recursion = 3

call                        | action
Hanoi(3,A,C,B)              | calls Hanoi(2,A,B,C)
  Hanoi(2,A,B,C)            | calls Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)          | prints: move disk 1 from A to C, returns
  Hanoi(2,A,B,C)            | prints: move disk 2 from A to B, calls Hanoi(1,C,B,A)
    Hanoi(1,C,B,A)          | prints: move disk 1 from C to B, returns
  Hanoi(2,A,B,C)            | returns
Hanoi(3,A,C,B)              | prints: move disk 3 from A to C, calls Hanoi(2,B,C,A)
  Hanoi(2,B,C,A)            | calls Hanoi(1,B,A,C)
    Hanoi(1,B,A,C)          | prints: move disk 1 from B to A, returns
  Hanoi(2,B,C,A)            | prints: move disk 2 from B to C, calls Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)          | prints: move disk 1 from A to C, returns
 

**Try it yourself.** Change the number of disks and count the moves.

In [24]:
%%writefile ProvaHanoi.java
public class ProvaHanoi
{
    public static void main(String[] args)
    {
        int n = 4;        // <-- change the number of disks
        Hanoi.reset();
        Hanoi.showMoves = true;
        Hanoi.hanoi(n, "A", "C", "B");
        Hanoi.showMoves = false;
        System.out.printf("%d moves = 2^%d - 1 = %d\n", Hanoi.moves, n, (1 << n) - 1);
    }
}

Writing ProvaHanoi.java


In [25]:
!javac ProvaHanoi.java && java ProvaHanoi

1. move disk 1 from A to B
2. move disk 2 from A to C
3. move disk 1 from B to C
4. move disk 3 from A to B
5. move disk 1 from C to A
6. move disk 2 from C to B
7. move disk 1 from A to B
8. move disk 4 from A to C
9. move disk 1 from B to C
10. move disk 2 from B to A
11. move disk 1 from C to A
12. move disk 3 from B to C
13. move disk 1 from A to B
14. move disk 2 from A to C
15. move disk 1 from B to C
15 moves = 2^4 - 1 = 15
